# 03 · Tu caso

Hasta acá construiste el pipeline sobre un caso de ejemplo. Ahora lo apuntamos a
**tu archivo y tu web**.

> ## ⚠️ Este cuaderno corre en tu máquina, no en Colab
>
> Por dos razones, y las dos importan:
>
> 1. **Tu web es interna.** Colab no la alcanza.
> 2. **Vas a querer ver el navegador trabajando.** Un cuaderno solo te puede
>    mostrar capturas estáticas; la automatización andando en vivo, no.

```bash
git clone https://github.com/GEJ1/data_entry_automatizado.git
cd data_entry_automatizado
python3.10 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
playwright install chromium
jupyter notebook 03_tu_caso.ipynb
```

---

## Los checkpoints acá son distintos

En `02` yo sabía la respuesta correcta y podía verificarte. **Acá no.** No conozco
tu documento ni tu web.

Así que los checkpoints cambian de forma: pasan a ser **preguntas que te
respondés vos**. Suena más flojo y no lo es — es exactamente lo que vas a tener
que hacer cada vez que cambie el documento de origen.

In [ ]:
import subprocess, sys
from pathlib import Path

assert Path("pipeline/contratos.py").exists(), (
    "Corré este cuaderno desde la raíz del repo clonado en tu máquina.")

PY = sys.executable

def correr(comando, mostrar=True):
    r = subprocess.run(comando, shell=True, capture_output=True, text=True)
    salida = (r.stdout + r.stderr).rstrip()
    if mostrar:
        print(salida)
    return salida

def preguntas(titulo, items):
    """Checkpoint de autoevaluación: respondé con True/False."""
    print(titulo); print("=" * len(titulo))
    faltan = [t for t, ok in items if not ok]
    for t, ok in items:
        print(f"  [{'✓' if ok else ' '}] {t}")
    if faltan:
        print(f"\n  ⏳ te faltan {len(faltan)}. No sigas todavía.")
    else:
        print("\n  ✅ listo, seguí")
    return not faltan

print("listo ·", Path.cwd())

---
# Paso 1 · El andamio

Esta celda te crea `mi_caso/` con los esqueletos de las tres piezas que vas a
escribir. **No pisa nada si ya existe**, así que la podés correr tranquilo.

In [ ]:
from pathlib import Path

BASE = Path("mi_caso")
BASE.mkdir(exist_ok=True)

ARCHIVOS = {}

ARCHIVOS["extractor.py"] = '''"""
TU EXTRACTOR. Cumple el contrato de pipeline/contratos.py.

Recordá la regla de oro: esto NO sabe de negocio. Solo abre el archivo y pone
texto en cabecera y tablas. Nada de fechas, números ni validaciones.

Para enchufarlo, agregalo al registro en pipeline/extractores/__init__.py:
    from mi_caso.extractor import MiExtractor
    EXTRACTORES = [ExtractorPDF(), ExtractorDOCX(), MiExtractor()]
"""
from pathlib import Path
from typing import Iterator

from pipeline.contratos import FichaCruda


class MiExtractor:
    formatos = (".pdf",)     # TODO: la extensión de TU archivo
    nombre = "mi_formato"

    def extraer(self, ruta: Path) -> Iterator[FichaCruda]:
        ruta = Path(ruta)
        # TODO: recorrer el documento y cortar por el encabezado de cada unidad.
        #
        # Acordate de las tres trampas del caso de ejemplo, que suelen repetirse:
        #   1. ¿las fichas se derraman a varias páginas sin repetir el encabezado?
        #   2. ¿cómo distinguís una tabla de otra? (mirá SUS encabezados)
        #   3. ¿hay filas que son títulos y no datos? (celdas combinadas)
        yield FichaCruda(
            cabecera={},          # los datos sueltos de arriba de la ficha
            tablas={},            # {"nombre_seccion": [ {col: texto}, ... ]}
            origen=ruta.name)
'''

ARCHIVOS["dominio.py"] = '''"""
TU DOMINIO: qué significan los datos de TU documento.

Los normalizadores genéricos (fechas, montos, identificadores) los reusás de
pipeline/dominio/normalizadores.py: el data entry siempre pelea contra los
mismos problemas. Lo que se reescribe es lo de acá.
"""
from datetime import date

from pydantic import BaseModel

from pipeline.contratos import FichaCruda
from pipeline.dominio.normalizadores import (
    clave_columna, parse_fecha, parse_monto,
)

# Encabezado del documento -> campo del modelo.
# Las claves van normalizadas con clave_columna(): tolera tildes y mayúsculas.
# Si el documento cambia los encabezados, se toca ACÁ y en ningún otro lado.
COLUMNAS = {
    # "fecha de carga": "fecha",
    # "razon social": "nombre",
}


class MiFila(BaseModel):
    """TODO: los campos de UNA fila de tu documento."""
    # fecha: date | None
    # nombre: str
    problemas: list[str] = []


class MiUnidad(BaseModel):
    """TODO: la unidad que se repite en tu documento (cliente, factura, paciente...)."""
    identificador: str
    filas: list[MiFila] = []
    descartadas: int = 0
    origen: str = ""

    @property
    def problemas(self) -> list[str]:
        p = []
        for i, f in enumerate(self.filas, start=1):
            p += [f"fila {i}: {x}" for x in f.problemas]
        return p

    @classmethod
    def from_ficha(cls, ficha: FichaCruda) -> "MiUnidad":
        # TODO: mapear la ficha cruda a este modelo
        return cls(identificador="?", origen=ficha.origen)
'''

ARCHIVOS["mapeo_web.yaml"] = '''# TU WEB.
# Este archivo es lo único que se toca para apuntar el pipeline a otro sistema.

base_url: "http://localhost:8000"    # TODO
timeout_ms: 5000

formularios:

  mi_formulario:
    # {cuit} y demás se reemplazan con lo que pongas en ItemDeCarga.busqueda
    url_busqueda: "/buscar?q={cuit}"      # TODO

    selectores:
      sin_resultados: "#sin-resultados"   # TODO: cómo sabés que no encontró nada
      fila: "tr.fila"                     # TODO: las filas de la tabla
      celda_fecha: "td.fecha"             # TODO: lo que identifica la fila
      celda_quien: "td.quien"             # TODO
      link_editar: "a.editar"             # TODO: el lápiz. Su href es el id.
      boton_guardar: "#guardar"           # TODO
      aviso_guardado: "#aviso"            # TODO: cómo confirma que guardó

    # LISTA BLANCA: los ÚNICOS campos que el pipeline puede escribir.
    # Escribir algo que no esté acá aborta la fila. Sé tacaño con esta lista.
    campos:
      # mi_campo: "#mi_campo"
'''

ARCHIVOS["verdad.json"] = '''[
  {
    "cabecera": {},
    "tablas": {}
  }
]
'''

ARCHIVOS["README.md"] = '''# mi_caso

Tu adaptación. **No commitees este directorio**: puede tener selectores internos
y datos de tu organización. Ya está en el `.gitignore`.

| Archivo | Qué es |
|---|---|
| `extractor.py` | Lee TU formato. No sabe de negocio. |
| `dominio.py` | Qué significan TUS datos. |
| `mapeo_web.yaml` | TU web: URLs, selectores y lista blanca. |
| `verdad.json` | Tu ground truth, escrito a mano. |
| `muestra.*` | Tu archivo de ejemplo ANONIMIZADO (ponelo vos). |
'''

creados, existentes = [], []
for nombre, contenido in ARCHIVOS.items():
    destino = BASE / nombre
    if destino.exists():
        existentes.append(nombre)
    else:
        destino.write_text(contenido, encoding="utf-8")
        creados.append(nombre)

# que no se te escape al repo
gi = Path(".gitignore")
if "mi_caso/" not in gi.read_text(encoding="utf-8"):
    gi.write_text(gi.read_text(encoding="utf-8").rstrip() + "\n\n# tu adaptación\nmi_caso/\n",
                  encoding="utf-8")
    print("mi_caso/ agregado al .gitignore")

print("creados  :", creados or "(ninguno)")
print("ya había :", existentes or "(ninguno)")

---
# Paso 2 · Tu documento de muestra

Acá aparece la primera diferencia grande con el curso: **vos no tenés datos
falsos, tenés un documento real con datos sensibles**.

## La regla

> No desarrolles contra datos reales, y **no los pongas nunca en el repo.**

## Qué hacer

1. **Tomá UN documento** representativo. Uno, no cien.
2. **Anonimizalo**: reemplazá nombres, identificadores y montos por valores
   inventados, **manteniendo la forma**. Si los CUIT tienen dígito verificador,
   que los falsos también lo tengan válido, o vas a estar depurando un problema
   que inventaste vos.
3. Guardalo como `mi_caso/muestra.pdf` (o la extensión que sea).
4. **Anotá las rarezas que veas** mientras lo mirás. Esa lista es oro: son tus
   trampas.

## Y después, el generador

Cuando el pipeline ande, escribí un generador de documentos falsos como el de
`demo/`. Parece una desviación y es la mejor inversión del proyecto: te da casos
difíciles a pedido, te deja probar a escala, y —sobre todo— te habilita el ground
truth del paso 4.

In [ ]:
from pathlib import Path

muestras = [p for p in Path("mi_caso").glob("muestra.*")]
preguntas("Paso 2 · tu documento de muestra", [
    ("hay un archivo mi_caso/muestra.*", bool(muestras)),
    ("está anonimizado (sin datos reales)", False),   # TODO: poné True cuando sea cierto
    ("anotaste las rarezas que viste", False),        # TODO
])
print("\nencontrado:", [str(m) for m in muestras] or "todavía nada")

---
# Paso 3 · Tus reglas sucias

Antes de tocar código: **abrí tu documento y buscá las columnas mentirosas.**

Para cada columna que no sea obviamente limpia, escribí:

| Pregunta | Ejemplo del curso |
|---|---|
| ¿Qué formatos aparecen? | 10 formas distintas de escribir una antigüedad |
| ¿Qué hago con lo que no entiendo? | plazo `3060` → campo `None` + marca, la fila igual se carga |
| ¿Qué hago con lo inválido pero identificable? | CUIT con verificador malo → se carga y se marca |
| ¿Hay filas que se excluyen legítimamente? | `Inactivo = Sí` → se descarta, **pero se cuenta** |
| ¿Hay valores relativos? | `2 meses` → anclado a la emisión, **nunca a hoy** |

Las tres decisiones que **siempre** hay que tomar explícitamente:

1. **Lo dudoso, ¿bloquea o se marca?** Casi siempre: se marca y sigue. Un registro
   que desaparece en silencio es peor que uno marcado.
2. **Lo vacío y lo ilegible, ¿son lo mismo?** No. `""` es un vacío legítimo;
   `"treinta"` en una columna numérica es un problema. Mezclarlos te esconde
   errores reales entre cientos de celdas vacías.
3. **Lo relativo, ¿contra qué se ancla?** Contra una fecha del documento. Si usás
   `date.today()`, el mismo archivo da resultados distintos según el día.

> ### 🤖 Pedíselo a Claude
>
> ```
> Estas son las columnas de mi documento y 10 valores reales de cada una
> [pegar]. Para cada columna, decime qué formatos distintos detectás, cuáles son
> ambiguos, y proponeme una regla de normalización. Marcá explícitamente los casos
> donde haya que tomar una decisión de negocio en vez de una técnica.
> ```

In [ ]:
preguntas("Paso 3 · tus reglas", [
    ("listaste las columnas sucias de tu documento", False),   # TODO
    ("decidiste qué hacer con lo ilegible en cada una", False),
    ("distinguiste 'vacío' de 'no se entiende'", False),
    ("si hay valores relativos, elegiste el ancla", False),
])

---
# Paso 4 · Tu ground truth, a mano

**La habilidad más transferible del curso.**

En el ejemplo, el ground truth lo generaba el propio generador. Vos todavía no
tenés generador. Pero no lo necesitás para empezar: **escribí a mano lo que
esperás de 2 o 3 fichas.**

Sí, es tedioso. Es mucho menos tedioso que descubrir dentro de tres semanas que
venías comiéndote una fila de cada cuarenta.

Abrí `mi_caso/verdad.json` y completá, copiando **literalmente** lo que ves en el
documento —sin interpretar nada, que eso es tarea del dominio:

```json
[
  {
    "cabecera": {"Cliente": "ACME SA", "CUIT": "30709204595"},
    "tablas": {
      "referencias": [
        {"Fecha": "12/5/2006", "Plazo": "30"}
      ]
    }
  }
]
```

In [ ]:
import json
from pathlib import Path

ruta = Path("mi_caso/verdad.json")
verdad = json.loads(ruta.read_text(encoding="utf-8"))
completo = any(f.get("cabecera") for f in verdad)

print(f"fichas en tu verdad.json: {len(verdad)}")
if not completo:
    print("→ Todavía está la plantilla vacía. Completá 2 o 3 fichas a mano.")
else:
    for f in verdad:
        filas = sum(len(v) for v in f.get("tablas", {}).values())
        print(f"   {f['cabecera']}  ({filas} filas)")

preguntas("\nPaso 4 · ground truth", [
    ("escribiste a mano 2-3 fichas esperadas", completo),
    ("copiaste el texto LITERAL, sin interpretarlo", False),   # TODO
    ("incluiste al menos una ficha con un caso raro", False),  # TODO
])

### Cómo verificar tu extractor contra eso

Cuando tengas el extractor, la comparación es la misma función que usa el curso:

```python
from tests.verificar_extractor import _diferencias
from mi_caso.extractor import MiExtractor

obtenidas = [f.a_dict() for f in MiExtractor().extraer("mi_caso/muestra.pdf")]
for esperada, obtenida in zip(verdad, obtenidas):
    for d in _diferencias(esperada, {k: v for k, v in obtenida.items() if k != "origen"}):
        print(" ·", d)
```

---
# Paso 5 · Tu extractor

Escribí `mi_caso/extractor.py`. Las tres preguntas que casi siempre aparecen,
sea cual sea el formato:

1. **¿Dónde empieza y termina una unidad?** Buscá un encabezado que se repita.
   Cortá por ahí, no por página ni por cantidad de filas.
2. **¿Se derrama?** Si una unidad puede seguir en la página siguiente, procesá el
   documento entero como un flujo. Nunca página por página.
3. **¿Cómo distinguís una tabla de otra?** Por sus propios encabezados de columna,
   no por el título de sección ni por la posición.

Y la regla que no se negocia: **el extractor devuelve strings.** Si te encontrás
escribiendo `int(...)` o `datetime(...)` ahí adentro, eso va en el dominio.

> ### 🤖 Pedíselo a Claude
>
> ```
> Escribí un extractor para mi documento que cumpla el contrato de
> pipeline/contratos.py. La unidad que se repite es [describir], y el encabezado que
> la abre es [pegar un ejemplo]. Las tablas son [describir]. El extractor no debe
> interpretar nada: solo texto crudo en cabecera y tablas.
> ```

In [ ]:
try:
    from mi_caso.extractor import MiExtractor
    ex = MiExtractor()
    print("formatos que maneja:", ex.formatos)
    muestras = list(Path("mi_caso").glob("muestra.*"))
    if muestras:
        fichas = list(ex.extraer(muestras[0]))
        print(f"fichas extraídas: {len(fichas)}")
        if fichas:
            print("cabecera de la primera:", fichas[0].cabecera)
            for seccion, filas in fichas[0].tablas.items():
                print(f"   {seccion}: {len(filas)} filas")
    else:
        print("poné tu archivo en mi_caso/muestra.*")
except ImportError as e:
    print("todavía no se puede importar:", e)

---
# Paso 6 · Tu web

Ahora los selectores. Abrí tu web, entrá al formulario que llenás a mano y
**usá el inspector del navegador** (click derecho → Inspeccionar).

## Lo que tenés que encontrar

| Qué | Para qué |
|---|---|
| URL de búsqueda | llegar al registro. ¿Se puede armar con un parámetro? |
| Cómo se ve "no encontrado" | para no esperar de gusto |
| Las filas de la tabla | un selector que las agarre a todas |
| Las celdas que identifican la fila | fecha + quién, o lo que corresponda |
| El link de editar | **su `href` es el id**. No lo calcules: cosechalo. |
| El botón de guardar | |
| Cómo confirma que guardó | para verificar, no para asumir |

## Y lo más importante: qué NO tocar

Recorré el formulario campo por campo y anotá **cuáles no debe escribir el
pipeline jamás**. Campos de otro equipo, de otro sistema, calculados, legales.

Después escribí la lista blanca **al revés**: no "todo menos esos", sino
**solamente estos**. Si mañana agregan un campo nuevo al formulario, con una
lista negra quedaría desprotegido; con una lista blanca, no.

> ### 🤖 Pedíselo a Claude
>
> ```
> Este es el HTML del formulario donde cargo los datos [pegar]. Armame
> el bloque de config/mapeo_web.yaml con los selectores: cómo buscar, cómo
> identificar cada fila, dónde está el link de edición, el botón de guardar y la
> confirmación. Y proponeme la lista blanca de campos escribibles, dejando afuera
> todo lo que parezca de otro sistema o calculado.
> ```

In [ ]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(Path("mi_caso/mapeo_web.yaml").read_text(encoding="utf-8"))
form = (cfg.get("formularios") or {}).get("mi_formulario") or {}
campos = form.get("campos") or {}
sels = form.get("selectores") or {}

print("base_url :", cfg.get("base_url"))
print("campos en la lista blanca:", list(campos) or "(ninguno todavía)")
print("selectores definidos     :", list(sels))

preguntas("\nPaso 6 · tu web", [
    ("cambiaste base_url por la tuya", cfg.get("base_url") != "http://localhost:8000"),
    ("definiste la lista blanca de campos", bool(campos)),
    ("hiciste UNA fila a mano y anotaste cada paso", False),   # TODO
    ("identificaste los campos que NO hay que tocar", False),  # TODO
])

---
# Paso 7 · La carga · ⚠️ en tu terminal

Igual que en el curso, esto **no va en el cuaderno**. Es el momento donde ves la
automatización trabajando, y eso hay que mirarlo en vivo.

## El orden, que no es negociable

**1 · Simulado** — sin navegador. Verifica claves, conflictos y lista blanca.

```bash
python -m pipeline.cli cargar mi_caso/salida.jsonl --simular --mapeo mi_caso/mapeo_web.yaml
```

**2 · Dry-run de UNA fila, mirándola** — llena el formulario y **no guarda**.

```bash
python -m pipeline.cli cargar mi_caso/salida.jsonl --dry-run --limite 1 \
    --ver --lento 600 --mapeo mi_caso/mapeo_web.yaml
```

Quedate mirando. ¿Llenó los campos que esperabas? ¿Tocó alguno que no? ¿Encontró
la fila correcta? **No sigas hasta que esta corrida te convenza.**

**3 · Una fila de verdad** — y verificá a mano en la web que quedó bien.

```bash
python -m pipeline.cli cargar mi_caso/salida.jsonl --limite 1 --mapeo mi_caso/mapeo_web.yaml
```

**4 · Recién ahí, el lote.**

```bash
python -m pipeline.cli cargar mi_caso/salida.jsonl --mapeo mi_caso/mapeo_web.yaml
```

> `--ver` abre el navegador con ventana · `--lento 600` frena cada acción para
> que se siga con la vista · `--limite N` corta después de N filas.

## El checkpoint antes de soltar el lote

Este es el que importa. Respondé honestamente: cambiá a `True` solo lo que
efectivamente hiciste.

In [ ]:
listo = preguntas("¿Estás listo para cargar el lote completo?", [
    ("la reconciliación cierra contra el documento", False),
    ("la hoja Problemas está vacía (o revisaste cada fila)", False),
    ("hiciste el dry-run y llenó lo que esperabas", False),
    ("cargaste UNA fila de verdad y la verificaste en la web", False),
    ("la lista blanca deja afuera todo lo que no hay que tocar", False),
    ("sabés qué hace el pipeline si dos filas son idénticas", False),
    ("probaste cortar con Ctrl-C y retomar sin duplicar", False),
])

if not listo:
    print("\n   Cada casillero sin marcar es una forma conocida de romper datos ajenos.")

---
# Cuando algo falle

Porque va a fallar, y la pregunta es dónde te enterás.

| Síntoma | Dónde mirar |
|---|---|
| Faltan registros | la **reconciliación** de `extraer`: ¿cierran los números? |
| Un dato llega mal | la hoja **Detalle** del Excel, y de ahí a la regla del dominio |
| "No encontré la fila" | tus selectores, y si lo que identifica la fila es único |
| "N filas matchean" | tenés claves duplicadas: **está bien que no cargue** |
| Se cargó algo que no iba | la **lista blanca**. Achicala. |
| El read-back falla | la web transforma lo que escribís (formato de fecha, decimales) |

## Las cinco cosas que te llevás

1. **Las costuras primero.** El contrato antes que el parser.
2. **Ground truth.** "Le pasé tres archivos y anduvo" no es una verificación.
3. **Lo dudoso se marca, no se esconde.** Y no bloquea.
4. **Lista blanca, no lista negra.** Y verificada, no declamada.
5. **Ante la duda, no cargar.** Automatizar bien no es cargar todo: es saber qué
   no cargar.

---

**El repo:** https://github.com/GEJ1/data_entry_automatizado